Step 3 : Data Modeling

In [1]:
import pandas as pd
import numpy as np

Load Cleaned Dataset

In [2]:
df = pd.read_csv("cleaned_budget_data.csv")

df.head()

,Date,Department,Category,Region,Budget Amount,Actual Amount,Payment Method,Transaction ID,Variance,Variance %,Status
0,2023-05-11,Sales,Travel,North,126096,43048,Card,TXN100000,-83048,-65.860931,Within Budget
1,2023-11-11,Marketing,Salaries,East,19702,87896,Bank Transfer,TXN100001,68194,346.127297,Over Budget
2,2021-05-02,It,Training,Central,108523,103632,Card,TXN100002,-4891,-4.506879,Within Budget
3,2022-04-12,Marketing,Salaries,North,114711,105574,Cash,TXN100003,-9137,-7.965234,Within Budget
4,2021-11-27,Sales,Utilities,Central,121895,64314,UPI,TXN100004,-57581,-47.238197,Within Budget


Create Budget Utilization %

This shows how much of the budget was used.

Formula:

Budget Amount
Actual Amount   ×100

In [3]:
df['Budget Utilization %'] = (
    df['Actual Amount']
    /
    df['Budget Amount']
) * 100

Create Risk Level

Categorize departments based on overspending.

In [4]:
def risk_level(x):

    if x <= 100:
        return "Low"

    elif x <= 110:
        return "Medium"

    else:
        return "High"

In [5]:
df['Risk Level'] = df['Budget Utilization %'].apply(risk_level)

Department Performance Score

Create department summary.

In [6]:
department_summary = df.groupby('Department').agg({
    'Budget Amount':'sum',
    'Actual Amount':'sum',
    'Variance':'sum'
}).reset_index()

Department Budget Utilization

In [7]:
department_summary['Budget Utilization %'] = (
    department_summary['Actual Amount']
    /
    department_summary['Budget Amount']
) * 100

Department Ranking

Lower variance = better performance.

In [8]:
department_summary['Department Rank'] = (
    department_summary['Variance']
    .rank(ascending=True)
)

Category Performance Model

In [9]:
category_summary = df.groupby('Category').agg({
    'Budget Amount':'sum',
    'Actual Amount':'sum',
    'Variance':'sum'
}).reset_index()

Category Ranking

In [10]:
category_summary['Category Rank'] = (
    category_summary['Variance']
    .rank(ascending=True)
)

Overspending Detection Model

Identify all overspending transactions.

In [11]:
overspending = df[
    df['Actual Amount']
    >
    df['Budget Amount']
]

High-Risk Transaction Detection

In [15]:
high_risk = df[
    df['Budget Utilization %']
    >
    120
]

Monthly Performance Model

In [21]:
print(df.columns)

Index(['Date', 'Department', 'Category', 'Region', 'Budget Amount',
       'Actual Amount', 'Payment Method', 'Transaction ID', 'Variance',
       'Variance %', 'Status', 'Budget Utilization %', 'Risk Level', 'Month'],
      dtype='object')


In [19]:
df['Date'] = pd.to_datetime(df['Date'])

df['Month'] = df['Date'].dt.month_name()

In [20]:
monthly_summary = df.groupby('Month').agg({
    'Budget Amount':'sum',
    'Actual Amount':'sum',
    'Variance':'sum'
}).reset_index()

Monthly Efficiency Score

Formula:

100−∣Variance%∣

In [23]:
monthly_summary['Variance %'] = (
    monthly_summary['Variance']
    /
    monthly_summary['Budget Amount']
) * 100

In [24]:
monthly_summary['Efficiency Score'] = (
    100 -
    abs(monthly_summary['Variance %'])
)

Savings Opportunity Model

In [25]:
savings = df[
    df['Variance']
    <
    0
]

Generate Final Dashboard Tables
Department Dashboard Table

In [26]:
department_summary.head()

,Department,Budget Amount,Actual Amount,Variance,Budget Utilization %,Department Rank
0,Finance,132947149,144084529,11137380,108.377299,1.0
1,Hr,137486926,155554390,18067464,113.141223,5.0
2,It,125771001,139663026,13892025,111.045491,2.0
3,Marketing,131939479,151816098,19876619,115.064952,6.0
4,Operations,132622213,147803987,15181774,111.447384,3.0


Category Dashboard Table

In [27]:
category_summary.head()

,Category,Budget Amount,Actual Amount,Variance,Category Rank
0,Infrastructure,127813870,140825422,13011552,2.0
1,Marketing,130224132,146525696,16301564,4.0
2,Salaries,127131160,147744247,20613087,6.0
3,Training,132399473,145010926,12611453,1.0
4,Travel,140009003,154900418,14891415,3.0


Export Model Outputs
Department Summary

In [30]:
department_summary.to_csv(
    "department_summary.csv",
    index=False
)

Category Summary

In [31]:
category_summary.to_csv(
    "category_summary.csv",
    index=False
)

Monthly Summary

In [32]:
monthly_summary.to_csv(
    "monthly_summary.csv",
    index=False
)